# GCOL-OBS-0.3.2.1.1-PATCH.10H.0R
## Provenance and SPW mapping reconciliation — FAST / metadata-only

**Decision scope:** reconcile historical SPW naming and recover missing provenance, without opening line amplitudes. The files from `PATCH.10H.0` and `PATCH.10H.1` are imported **unchanged**. The X24 **SPW25 raw spectral FITS HEADER ONLY** and X24 SPW25 MFS header were additionally read. Full raw SPW25 cube byte SHA-256 was computed for integrity; pixel values were **not** decoded or inspected.

- ALMA project `2015.1.00147.S`; source `NGC1333_IRAS4A`; member data X22, X24.
- Frozen windows: **6.40–7.00 km/s**, **7.70–8.30 km/s**.
- Frozen primary SPW labels: `[27,29,31,33,35,37]`; SPW25 initially **not** primary.
- This work **does not** certify a foreground layer, an ordering of layers, or GVH physics (`GVH_USED=False`).
- A new proposed mapping may **not** overwrite the preregistration. A separately approved versioned amendment is needed before the signal test.

**Scientific note:** FITS `RESTFRQ` is an instrument/reduction header value; a match to a molecular transition is **not** independently proved by the header. Names `H2CO`/`HNC` below reproduce the *original protocol's labels*, not new line identifications.

In [1]:
from pathlib import Path
import json, hashlib, math
import numpy as np
import pandas as pd
from collections import defaultdict

# Portable: place notebook in same directory as `inputs/`, or inside bundle root.
try: ROOT = Path.cwd()
except NameError: ROOT = Path('.')
if not (ROOT / 'inputs' / 'PATCH10H0_status.json').is_file():
    ROOT = Path('/mnt/data/GCOL_PATCH10H0R_PACKAGE')
IN=ROOT/'inputs'
assert IN.is_dir(), 'Missing bundled inputs: extract reproducibility ZIP next to notebook.'
print('SOURCE ROOT',ROOT)
CHECKS={}
def check(name, condition):
    good=bool(condition)
    CHECKS[name]=good
    if not good: raise AssertionError(f'check failed: {name}')
    return good

def sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(4<<20),b''):h.update(chunk)
    return h.hexdigest()

def read_json(n):return json.loads((IN/n).read_text())


SOURCE ROOT /mnt/data/GCOL_PATCH10H0R_PACKAGE


## 1. Immutable historical state: H0 and H1

Import the original frozen contract and the recorded H0 failure **without promoting** any gate. The H1 continuum-only geometry is evidence of a separate partial step, not proof of H0 success.

In [2]:
contract=read_json('PATCH10H0_preregistration_contract.json')
h0=read_json('PATCH10H0_status.json')
h1=read_json('GCOL_OBS_0_3_2_1_1_PATCH10H1_Continuum_Geometry_and_Spectral_Metadata_Preflight_FAST_results.json')
source=read_json('raw_header_source_manifest.json')
headers=pd.read_csv(IN/'PATCH10H0_fits_header_audit.csv')
required=pd.read_csv(IN/'PATCH10H0_required_set_audit.csv')
check('h0_failed',h0['PATCH10H0_PASS'] is False)
check('h0_not_authorized',h0['NEXT_PATCH_AUTHORIZED'] is False)
check('h0_readme_missing',h0['README_PRESENT'] is False)
check('h0_x22_missing_cont',h0['X22_REQUIRED_SET_PASS'] is False)
check('h0_x24_complete',h0['X24_REQUIRED_SET_PASS'] is True)
check('h1_partial',h1['PATCH10H0_PASS'] is False and h1['X24_CONTINUUM_GEOMETRY_MEASURED'] is True)
check('project_matches',contract['project_code']=='2015.1.00147.S' and contract['source']=='NGC1333_IRAS4A')
check('frozen_windows_preserved',contract['velocity_windows_kms']=={'core':[6.4,7.0],'foreground_test':[7.7,8.3]})
check('frozen_spw25_secondary',contract['spw25_primary_test'] is False)
check('h0_12_primary_headers',len(headers[(headers.product_class=='cube_pbcor') & (headers.spw.isin(contract['primary_spws']))])==12)
print('H0 verdict:',h0['PATCH10H0_DECISION'],'— DO NOT CHANGE')
print('H1 verdict:',h1['DECISION'])
print('Historical required set:\n',required.to_string(index=False))


H0 verdict: ABSTAIN_ARCHIVE_OR_PROVENANCE_NOT_READY — DO NOT CHANGE
H1 verdict: PARTIAL_REAL_X24_CONTINUUM_GEOMETRY_ONLY__FULL_GATE_BLOCKED
Historical required set:
 member     primary_spws_present primary_spws_missing  continuum_pbcor_count  required_set_pass
   X22 [27, 29, 31, 33, 35, 37]                   []                      0              False
   X24 [27, 29, 31, 33, 35, 37]                   []                      1               True


## 2. Native FITS SPW25: read header cards only

FITS keyword values are extracted from a small header-card text snapshot. The original 195 MB cube is **not required** for local replay. Header snapshots preserve the original 80-character FITS card lines, with a provenance manifest including original Google Drive IDs and full-FITS SHA-256.

In [3]:
def parse_fits_cards(filename):
    d={}
    for line in (IN/filename).read_text().splitlines():
        if line[:8].strip()=='END':break
        if len(line)<10 or line[8:10]!='= ':continue
        key=line[:8].strip(); value=line[10:].split('/')[0].strip()
        if value.startswith("'"):d[key]=value.strip().strip("'").strip()
        elif value in {'T','F'}:d[key]=(value=='T')
        else:
            try:d[key]=float(value) if any(i in value.upper() for i in '.ED') else int(value)
            except ValueError:d[key]=value
    return d
cube25=parse_fits_cards('X24_spw25_cube_HEADER_ONLY_cards.txt')
mfs25=parse_fits_cards('X24_spw25_mfs_HEADER_ONLY_cards.txt')
check('spw25_cube_primary',cube25['SIMPLE'] is True and cube25['NAXIS']==4)
check('spw25_cube_nchan',cube25['NAXIS3']==478)
check('spw25_mfs_singleton',mfs25['NAXIS3']==1)
check('spw25_cube_frequency_axis',cube25['CTYPE3']=='FREQ' and cube25['CUNIT3']=='Hz')
check('spw25_cube_spectral_frame',cube25['SPECSYS']=='LSRK')
check('spw25_mfs_cube_rest_same',abs(cube25['RESTFRQ']-mfs25['RESTFRQ'])<1)
check('spw25_rest_363945894000',cube25['RESTFRQ']==363945894000.0)
check('spw25_cube_metadata_only',source['X24_SPW25_cube']['analysis_uses'].startswith('HEADER ONLY'))
check('spw25_cube_sha256_recorded',len(source['X24_SPW25_cube']['sha256_full_raw_fits'])==64)
print('SPW25 cube header:',{k:cube25.get(k) for k in ['NAXIS3','CRVAL3','CDELT3','CRPIX3','RESTFRQ','SPECSYS']})
print('SPW25 MFS RESTFRQ:',mfs25['RESTFRQ'])
print('Cube provenance SHA-256:',source['X24_SPW25_cube']['sha256_full_raw_fits'])


SPW25 cube header: {'NAXIS3': 478, 'CRVAL3': 363908428308.0, 'CDELT3': 122244.5992432, 'CRPIX3': 1.0, 'RESTFRQ': 363945894000.0, 'SPECSYS': 'LSRK'}
SPW25 MFS RESTFRQ: 363945894000.0
Cube provenance SHA-256: 4c1d7549d3cdc44527ef5c234ff45b4e0fa8205915bf2bf8691704252612f4da


## 3. Spectral-axis reconstruction, no samples

For frequency-axis headers, channel **centres** in Hz are
\[\nu(n)=\mathrm{CRVAL3}+(n-\mathrm{CRPIX3})\,\mathrm{CDELT3},\quad n=1,\dots,N.\]
The velocity conversion for **coverage only** is the radio nonrelativistic convention \(\nu_\mathrm{obs}=\nu_0(1-v/c)\), with \(c=299792.458\,\mathrm{km/s}\). This does not estimate gas velocity or read absorption intensity.

In [4]:
C_KMS=299792.458
cols=['member','spw','spectral_nchan','spectral_crval','spectral_cdelt','spectral_crpix','restfreq_Hz','spectral_cunit','spectral_ctype']
prim=headers.loc[(headers.product_class=='cube_pbcor')&(headers.spw.isin(contract['primary_spws'])),cols].copy()
prim['spw']=prim.spw.astype(int)
prim['spectral_nchan']=prim.spectral_nchan.astype(int)
prim['from_header_archive']=True
extra=pd.DataFrame([{'member':'X24','spw':25,'spectral_nchan':cube25['NAXIS3'],'spectral_crval':cube25['CRVAL3'],'spectral_cdelt':cube25['CDELT3'],'spectral_crpix':cube25['CRPIX3'],'restfreq_Hz':cube25['RESTFRQ'],'spectral_cunit':cube25['CUNIT3'],'spectral_ctype':cube25['CTYPE3'],'from_header_archive':False}])
allrows=pd.concat([prim,extra],ignore_index=True)
allrows['f_first_Hz']=allrows.spectral_crval+(1-allrows.spectral_crpix)*allrows.spectral_cdelt
allrows['f_last_Hz']=allrows.spectral_crval+(allrows.spectral_nchan-allrows.spectral_crpix)*allrows.spectral_cdelt
allrows['fmin_Hz']=allrows[['f_first_Hz','f_last_Hz']].min(axis=1)
allrows['fmax_Hz']=allrows[['f_first_Hz','f_last_Hz']].max(axis=1)
allrows['dv_abs_kms']=C_KMS*np.abs(allrows.spectral_cdelt/allrows.restfreq_Hz)
check('axis_13_rows',len(allrows)==13)
check('all_axes_valid',bool(allrows.eval('spectral_nchan > 1 and fmin_Hz < fmax_Hz and restfreq_Hz > 0').all()))
check('spw37_wide',bool((allrows[allrows.spw==37].dv_abs_kms > .8).all()))
check('spw25_x24_available',len(allrows[(allrows.member=='X24')&(allrows.spw==25)])==1)
check('spw25_x22_unavailable',len(allrows[(allrows.member=='X22')&(allrows.spw==25)])==0)
print(allrows[['member','spw','spectral_nchan','fmin_Hz','fmax_Hz','restfreq_Hz','dv_abs_kms']].to_string(index=False))


member  spw  spectral_nchan      fmin_Hz      fmax_Hz  restfreq_Hz  dv_abs_kms
   X22   27             478 3.625929e+11 3.626511e+11 3.626303e+11    0.101015
   X22   29             478 3.626986e+11 3.627569e+11 3.627360e+11    0.100986
   X22   31             958 3.642083e+11 3.643252e+11 3.642751e+11    0.100508
   X22   33             478 3.640657e+11 3.641240e+11 3.641032e+11    0.100607
   X22   35             958 3.517021e+11 3.518190e+11 3.517686e+11    0.104079
   X22   37            1918 3.497559e+11 3.516279e+11 3.507000e+11    0.834777
   X24   27             478 3.625928e+11 3.626511e+11 3.626303e+11    0.101061
   X24   29             478 3.626985e+11 3.627569e+11 3.627360e+11    0.101032
   X24   31             958 3.642083e+11 3.643252e+11 3.642751e+11    0.100537
   X24   33             478 3.640657e+11 3.641241e+11 3.641032e+11    0.100653
   X24   35             958 3.517021e+11 3.518190e+11 3.517686e+11    0.104109
   X24   37            1918 3.497558e+11 3.516281e+1

## 4. Original contract vs observed candidate mapping

For every target rest frequency, choose the SPW with matching `RESTFRQ` (1 MHz tolerance) where available; otherwise note only spectral coverage (never infer chemical identification solely from the frequency). Where a frozen SPW contains no matching target, the discrepancy is documented rather than silently changed.

In [5]:
def covers_windows(row,nu0_Hz):
    def yes(win):
        lo,hi=win
        a=nu0_Hz*(1-lo/C_KMS);b=nu0_Hz*(1-hi/C_KMS)
        return row.fmin_Hz<=min(a,b) and max(a,b)<=row.fmax_Hz
    return {k:bool(yes(v)) for k,v in contract['velocity_windows_kms'].items()}

records=[]
for spw_original,target in contract['expected_spw_mapping'].items():
    for idx,ghz in enumerate(target['rest_GHz_expected']):
        for member in ['X22','X24']:
            mr=allrows[allrows.member==member]
            nu0=float(ghz)*1e9
            match=[]
            for _,r in mr.iterrows():
                cover=covers_windows(r,nu0)
                if all(cover.values()):
                    diff_khz=abs(r.restfreq_Hz-nu0)/1e3
                    match.append((int(r.spw),float(diff_khz),r,cover))
            match.sort(key=lambda x:x[1])
            best=match[0] if match else None
            records.append(dict(member=member,original_spw=int(spw_original),label_from_frozen_contract=target['label'],target_ghz=ghz,rest_candidate_spw=best[0] if best else None,header_rest_difference_khz=round(best[1],3) if best else None,coverage_core=bool(best[3]['core']) if best else False,coverage_foreground=bool(best[3]['foreground_test']) if best else False,frequency_covered=best is not None,header_rest_match_1MHz=(best[1]<=1000) if best else False,original_assignment_confirmed=bool(best and best[0]==int(spw_original)),other_matching_spws=[int(m[0]) for m in match[1:]]))
mapdf=pd.DataFrame(records)
check('contract_target_rows14',len(mapdf)==14)
check('historical_spw_mismatch_all12_primary',sum(mapdf[mapdf.original_spw.isin(contract['primary_spws'])].original_assignment_confirmed)==0)
check('x24_all7_target_freqs_covered',bool(mapdf[mapdf.member=='X24'].frequency_covered.all()))
check('x22_six_of_seven_targets',int(mapdf[mapdf.member=='X22'].frequency_covered.sum())==6)
check('target_36394589_x24_spw25',int(mapdf[(mapdf.member=='X24')&(abs(mapdf.target_ghz-363.94589)<1e-6)].iloc[0].rest_candidate_spw)==25)
check('target_36394589_x22_missing',not bool(mapdf[(mapdf.member=='X22')&(abs(mapdf.target_ghz-363.94589)<1e-6)].iloc[0].frequency_covered))
check('doublet_in_spw31',bool((mapdf[mapdf.original_spw==37].rest_candidate_spw==31).all()))
check('spw37_no_frozen_target',not bool((mapdf.rest_candidate_spw==37).any()))
check('all_frozen_line_windows_covered_where_candidate',bool(mapdf[mapdf.frequency_covered][['coverage_core','coverage_foreground']].all().all()))
print(mapdf[['member','original_spw','label_from_frozen_contract','target_ghz','rest_candidate_spw','frequency_covered','header_rest_match_1MHz']].to_string(index=False))


member  original_spw label_from_frozen_contract  target_ghz  rest_candidate_spw  frequency_covered  header_rest_match_1MHz
   X22            27                       H2CO   351.76864                35.0               True                    True
   X24            27                       H2CO   351.76864                35.0               True                    True
   X22            29                        HNC   362.63030                27.0               True                    True
   X24            29                        HNC   362.63030                27.0               True                    True
   X22            31                       H2CO   362.73605                29.0               True                    True
   X24            31                       H2CO   362.73605                29.0               True                    True
   X22            33                       H2CO   363.94589                 NaN              False                   False
   X24          

## 5. Cross-member agreement, unresolved data requirements, and candidate amendment

Candidate spectral reassignment (NOT activated) links the **frequency** from the frozen contract to actual SPW IDs. For target 363.94589 GHz, X24 SPW25 is backed by the raw cube header and independent continuum MFS header; **X22 has no archived SPW25** and does not cover that target in the six primary cubes.

Pending: missing X22 continuum; empty original README; user approval of a preregistration-preserving, dated amendment; verification of molecule IDs from an external molecular spectroscopy catalog or official ALMA processing metadata. Never unblind line amplitudes until authorized scientific gate.

In [6]:
amendment={
 'patch':'GCOL-OBS-0.3.2.1.1-PATCH.10H.0R',
 'status':'PROPOSAL_ONLY__NOT_ADOPTED',
 'original_contract_is_frozen':True,
 'original_primary_spws':contract['primary_spws'],
 'original_spw25_primary_test':contract['spw25_primary_test'],
 'velocity_windows_kms':contract['velocity_windows_kms'],
 'method':'frequency-axis coverage and FITS RESTFRQ metadata only; molecular IDs not independently verified',
 'proposed_frequency_to_native_spw_X24':{
   '351.76864':35,'362.63030':27,'362.73605':29,
   '363.94589':25,'364.10325':33,'364.27514':31,'364.28888':31
 },
 'proposed_frequency_to_native_spw_X22':{
   '351.76864':35,'362.63030':27,'362.73605':29,
   '363.94589':None,'364.10325':33,'364.27514':31,'364.28888':31
 },
 'spw37':'Broad spectral range 349.756–351.628 GHz, RESTFRQ 350.700 GHz; no originally targeted transition in its coverage with frozen velocity windows.',
 'remaining_blockers':['X22 continuum absent from accessible ALMA directory','original README empty','X22 lacks the 363.94589 GHz SPW25 target','approved versioned amendment absent','independent molecular rest-line identifiers not certified'],
 'scientific_status':{'line_amplitude_inspected':False,'foreground_layer_certified':False,'layer_order_derived':False,'GVH_USED':False,'line_test_authorized':False}}
check('mapping_proposal_not_activated',amendment['status'].startswith('PROPOSAL'))
check('no_line_unblinding',amendment['scientific_status']['line_amplitude_inspected'] is False)
check('no_GVH',amendment['scientific_status']['GVH_USED'] is False)
check('no_change_velocity_windows',amendment['velocity_windows_kms']==contract['velocity_windows_kms'])
check('h0_gate_remains_failed',h0['NEXT_PATCH_AUTHORIZED'] is False)
print('PROPOSED ONLY. Remaining blockers:\n','\n'.join('- '+x for x in amendment['remaining_blockers']))


PROPOSED ONLY. Remaining blockers:
 - X22 continuum absent from accessible ALMA directory
- original README empty
- X22 lacks the 363.94589 GHz SPW25 target
- approved versioned amendment absent
- independent molecular rest-line identifiers not certified


## 6. Decision certificate

**Reconciliation of frequency metadata** is successful where FITS coverage exists, but the **provenance and preregistration** gate is NOT passed; the 10H.0 failure is preserved. Data status is explicitly `PARTIAL_METADATA_RECONCILIATION__ABSTAIN_LINE_TEST`. These passing assertions test the consistency of our audit, not the physical validity of an absorption claim.

In [7]:
result={
 'patch':'GCOL-OBS-0.3.2.1.1-PATCH.10H.0R',
 'decision':'PARTIAL_METADATA_RECONCILIATION__ABSTAIN_LINE_TEST',
 'original_H0_decision':h0['PATCH10H0_DECISION'],
 'original_H0_PASS':h0['PATCH10H0_PASS'],
 'original_NEXT_PATCH_AUTHORIZED':h0['NEXT_PATCH_AUTHORIZED'],
 'original_contract_SHA256':sha(IN/'PATCH10H0_preregistration_contract.json'),
 'archived_fits_header_audit_SHA256':sha(IN/'PATCH10H0_fits_header_audit.csv'),
 'X24_SPW25_RAW_CUBE_SHA256':source['X24_SPW25_cube']['sha256_full_raw_fits'],
 'X24_SPW25_RESTFRQ_GHz':cube25['RESTFRQ']/1e9,
 'X24_SPW25_NCHAN':cube25['NAXIS3'],
 'X24_SPW25_freq_range_GHz':list((allrows[(allrows.member=='X24')&(allrows.spw==25)][['fmin_Hz','fmax_Hz']].iloc[0]/1e9).to_numpy()),
 'frozen_contract_unchanged':True,
 'frequency_target_coverage_X22':int(mapdf[mapdf.member=='X22'].frequency_covered.sum()),
 'frequency_target_coverage_X24':int(mapdf[mapdf.member=='X24'].frequency_covered.sum()),
 'target_count_per_member':len(mapdf)//2,
 'original_12_primary_assignment_matches':int(mapdf.original_assignment_confirmed.sum()),
 'missing_X22_continuum':True,
 'original_README_empty':True,
 'target_36394589_X22_no_primary_coverage':True,
 'line_amplitude_inspected':False,
 'line_test_authorized':False,
 'foreground_layer_certified':False,
 'front_back_order_derived':False,
 'GVH_USED':False,
 'requested_followup':'Recover X22 continuum and missing README, inspect official SPW definitions, obtain explicit approval for a versioned metadata-only amendment, verify X22 coverage; do NOT unblind until resolved',
 'checks':CHECKS,
 'check_count':len(CHECKS),
 'all_checks_passed':all(CHECKS.values())
}
assert result['check_count']>=30
assert result['all_checks_passed']
( ROOT/'PATCH10H0R_results.json').write_text(json.dumps(result,indent=2,ensure_ascii=False,default=float))
( ROOT/'PATCH10H0R_proposed_mapping_amendment_NOT_ADOPTED.json').write_text(json.dumps(amendment,indent=2,ensure_ascii=False))
mapdf.to_csv(ROOT/'PATCH10H0R_frequency_mapping_evidence.csv',index=False)
allrows.to_csv(ROOT/'PATCH10H0R_header_only_spectral_axis_ranges.csv',index=False)
print(json.dumps({k:result[k] for k in ['decision','original_H0_PASS','frequency_target_coverage_X22','frequency_target_coverage_X24','target_count_per_member','check_count','all_checks_passed','line_test_authorized']},indent=2))


{
  "decision": "PARTIAL_METADATA_RECONCILIATION__ABSTAIN_LINE_TEST",
  "original_H0_PASS": false,
  "frequency_target_coverage_X22": 6,
  "frequency_target_coverage_X24": 7,
  "target_count_per_member": 7,
  "check_count": 38,
  "all_checks_passed": true,
  "line_test_authorized": false
}


## Four scientific levels and final lock

1. **GVH/CHH candidate:** not used; `GVH_USED=False`; CHH composition has no intrinsic derivative.
2. **Established physics / astronomy:** FITS headers, frequency axis and registered metadata, without assigning molecules from the header alone.
3. **Executed audit:** compare 14 target/member pairs, two immutable historical certificates, metadata-only SPW25 spectral cube audit; recompute all spectral coverages without sampling lines.
4. **Scientific verdict:** candidate SPW mapping is **reconciled as metadata**, but line testing remains **unauthorized**, and the original H0 failure remains in place. Proposed amendment must be explicitly approved and versioned; not a retroactive change to frozen analysis.